In [2]:
import models.utils as utils 
import models.agents.uni_random as uni_random
import models.agents.heuristics as heuristics
import models.agents.mcts as mcts
import analysis.analysis_utils as analysis_utils
import matplotlib.pylab as plt 
import seaborn as sns 
import itertools 
import pandas as pd
from tqdm import tqdm
from tqdm import trange
import random
import multiprocessing
import models.just_think.run_models 

import datetime
import importlib 
from itertools import chain
import constants
import numpy as np
import json

%matplotlib inline

In [4]:
""" 
Init constants
"""
importlib.reload(constants)
model2name = constants.MODEL2NAME
model2pth = constants.MODEL2PTH["think"]

ordered_models = ["ours", "random_terminal",]

"""
Load in games, and filter to include just the games from the live gameplay study
"""
games, idx2game, game2idx, game_stimuli = analysis_utils.process_game_stimuli(constants.THINK_STIMULI_PTH)
livegameplay_games = pd.read_csv(constants.PLAY_STIMULI_PTH)
livegameplay_games["stimuli_id"] = [analysis_utils.get_stimuli_id(row) for idx, row in livegameplay_games.iterrows()]
play_game_objs = []
played_game_ids = set()
for game_id in livegameplay_games["stimuli_id"]:
    game = game_stimuli[game_id]
    played_game_ids.add(game["index"])
    game["stimuli_id"] = game_id
    play_game_objs.append(game)

"""
Load in human data
"""

with open(f"../human-data/play-exp/human-v-human/final-play/final_agg.json", "r") as f: 
    human_gameplay_data = json.load(f)

think_human_df = pd.read_csv(constants.THINK_HUMAN_DATA)
# pulling out games according to Ced's game categories
all_game_types = {}
for i, entry in think_human_df.iterrows(): 
    game_type = entry.game_types
    game_id = entry.game_id
    if game_type not in all_game_types: all_game_types[game_type] = {game_id}
    else: all_game_types[game_type].add(game_id)

"""
Load in model preds
"""

model2res = {model: [] for model in ordered_models}
for model, model_pth in model2pth.items(): 
    res = analysis_utils.process_model_runs(model, model_pth, idx2game)
    if res is not None: model2res[model] = res


121
121
121
121


In [5]:
h_payoffs_games = {}

for i, (game_id, game) in enumerate(idx2game.items()): 
    # ordered_games.append(game)
    h_payoffs_games.setdefault(game, None)
    # humans
    hdf = think_human_df.loc[think_human_df.game_id == game]
    hdf = hdf.loc[hdf.task == "advantage"].reset_index()
    resps = eval(hdf['all_scores'][0])
    h_payoff = []
    for resp in resps: 
        pdraw = resp["draw_response"]
        pwin = analysis_utils.get_pwin(pdraw, resp["firstplayer_response"])
        pdraw /= 100
        pwin /= 100
        payoff = analysis_utils.compute_utility(pdraw, pwin)
        h_payoff.append(payoff)
    h_payoffs_games[game] = h_payoff

In [6]:
paramfit_dir = "../model-data/think-exp/param_fitting/"
import os

def load_paramfit_res(filename):
    param_setting = filename.split(".txt")[0] 
    param_setting = eval(param_setting)
    #print(f"{paramfit_dir}{filename}")
    with open(f"{paramfit_dir}{filename}", "r") as f: 
        res= f.readlines()
    res = eval(res[0])
    return res, param_setting

param_res = [load_paramfit_res(f) for f in os.listdir(paramfit_dir)]

In [7]:
len(param_res[0][0][0]['game_scores'])

50

In [8]:
''' 
Consider ablations of each, keeping the other fixed at 1.0
Three-way heatmap, averaging over one of the attributes
"Slices" 
'''
vals = np.arange(0, 2.2, 0.2)
attrs = ['center_weight', 'block_weight', 'connect_weight']
attr2viz = {'center_weight': 'Center Weight', 
            'block_weight': 'Block Weight', 
            'connect_weight': 'Connect Weight'}
default_attr_map = {attr: 1.0 for attr in attrs}
from copy import copy
# ablate each 

attr_ablations = []

default_attr_map = {attr: 1.0 for attr in attrs}

# Generate all the combinations we want to test
attr_configs = []

# All attributes enabled (baseline)
attr_configs.append(("All Components", copy(default_attr_map)))

# Single attribute ablations (each attribute at 0, others at 1)
for attr in attrs:
    ablated_map = copy(default_attr_map)
    ablated_map[attr] = 0.0
    attr_configs.append((f"Ablate {attr2viz[attr]}", ablated_map))

# Double attribute ablations (two at 0, one at 1)
for i, attr1 in enumerate(attrs):
    for attr2 in attrs[i+1:]:
        ablated_map = copy(default_attr_map)
        ablated_map[attr1] = 0.0
        ablated_map[attr2] = 0.0
        attr_configs.append((f"Only {attr2viz[next(a for a in attrs if a != attr1 and a != attr2)]}", ablated_map))

# Convert attribute maps to strings to use as keys for param2bootstrap_res
attr_config_keys = [(name, str(config)) for name, config in attr_configs]
keep_params = [val for k, val in attr_config_keys]

In [9]:
keep_params

["{'center_weight': 1.0, 'block_weight': 1.0, 'connect_weight': 1.0}",
 "{'center_weight': 0.0, 'block_weight': 1.0, 'connect_weight': 1.0}",
 "{'center_weight': 1.0, 'block_weight': 0.0, 'connect_weight': 1.0}",
 "{'center_weight': 1.0, 'block_weight': 1.0, 'connect_weight': 0.0}",
 "{'center_weight': 0.0, 'block_weight': 0.0, 'connect_weight': 1.0}",
 "{'center_weight': 0.0, 'block_weight': 1.0, 'connect_weight': 0.0}",
 "{'center_weight': 1.0, 'block_weight': 0.0, 'connect_weight': 0.0}"]

In [ ]:
importlib.reload(analysis_utils)
param2payoffs = {}
n_sim_participants = 20
n_simulations = 6
n_bootstrap = 100


np.random.seed(7)
random.seed(7)
for i, entry in enumerate(param_res):     
    res, param = entry 
    param2payoffs[str(param)] = []
    for _ in range(n_bootstrap): 
        game_payoffs = analysis_utils.process_novice_run_payoff(res, idx2game, 
                                                                n_sim_participants=n_sim_participants, 
                                                                n_simulations=n_simulations,bootstrap=True)
        param2payoffs[str(param)].append(game_payoffs)

IOStream.flush timed out


In [ ]:


param2bootstrap_res = {param_setting: [] for param_setting in param2payoffs}

np.random.seed(7)
random.seed(7)
from scipy.stats import pearsonr
    
for strap_idx in range(n_bootstrap): 
    # split participants per     
    # loop over param settings and compute fit 
    for param, res in param2payoffs.items(): 
        h_payoffs= []
        m_payoffs = []
        for game, m_payoff in res[strap_idx].items():
            m_payoffs.append(np.mean(m_payoff))
            h_payoffs.append(np.mean(h_payoffs_games[game]))
        r2 = pearsonr(h_payoffs, m_payoffs)[0] ** 2
        param2bootstrap_res[param].append(r2)


In [ ]:
''' 
Consider ablations of each, keeping the other fixed at 1.0
'''
attrs
default_attr_map = {attr: 1.0 for attr in attrs}
from copy import copy
# ablate each 
attr_ablations = []

base = np.mean(param2bootstrap_res[str(default_attr_map)])

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import ast
import pandas as pd

# help from Claude on viz

# Helper function to process the data
def process_data(data_dict, attrs):
    # Convert string dictionary keys to actual dictionaries
    processed_data = []
    for key, values in data_dict.items():
        # process to make sure floating points aren't crazy
        params = {k: round(float(v), 1) for k, v in ast.literal_eval(key).items()}
        #print(params)
        avg_value = np.mean(values)
        processed_data.append({**params, 'value': avg_value})
    
    return pd.DataFrame(processed_data)

def create_paired_heatmaps(data_dict, attrs, vals, max_val = 0.8, min_val = 0.38):
    # Process the data
    df = process_data(data_dict, attrs)
    
    # Create figure with three subplots
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    #fig.suptitle('Paired Attribute Heatmaps (R² Values)', fontsize=16)
    
    # Create heatmaps for each pair of attributes
    pairs = [
        (attrs[0], attrs[1]),  # center_weight vs block_weight
        (attrs[0], attrs[2]),  # center_weight vs connect_weight
        (attrs[1], attrs[2])   # block_weight vs connect_weight
    ]
    
    for idx, (attr1, attr2) in enumerate(pairs):
        # Create pivot table for this pair
        pivot_rows = []
        for val1 in vals:
            for val2 in vals:
                # Filter data for these specific values
                mask = (np.abs(df[attr1] - val1) < 0.1) & (np.abs(df[attr2] - val2) < 0.1)
                if mask.any():
                    avg_value = df[mask]['value'].mean()
                    pivot_rows.append({
                        attr1: val1,
                        attr2: val2,
                        'value': avg_value
                    })
        
        pivot_data = pd.DataFrame(pivot_rows)
        
        # Reshape data for heatmap
        if not pivot_data.empty:
            heatmap_data = pivot_data.pivot(
                index=attr1,
                columns=attr2,
                values='value'
            )
            
            # Create heatmap
            sns.heatmap(heatmap_data, 
                       ax=axes[idx],
                       cmap='viridis',
                       annot=True,
                       fmt='.2f',
                       vmin=min_val,
                       vmax=max_val)
                       #cbar_kws={'label': 'R² Value'})
            
            #axes[idx].set_title(f'{attr1} vs {attr2}')
            axes[idx].set_xticklabels([f'{x:.1f}' for x in heatmap_data.columns])
            axes[idx].set_yticklabels([f'{x:.1f}' for x in heatmap_data.index], rotation=0)
            
            axes[idx].set_xlabel(attr2viz[attr2], fontsize=18)
            axes[idx].set_ylabel(attr2viz[attr1], fontsize=18)
    
    plt.tight_layout()
    return fig

fig = create_paired_heatmaps(param2bootstrap_res, attrs, vals)
plt.tight_layout()
plt.savefig("param_fit_novice_just_think.png", dpi=400)